# Machine Learning using Sentinel 2 images through OpenEO and Forest Inventory data

In this document the Sentinel 2 images that cover the burned area after a forest fire are downloaded and a Machine Learning algorithm is trained to classify the vegetation species before the fire.

1. Connect to openEO and define the Cube
2. Define ML pipeline
3. Train ML algorithm locally
4. Predict values for the whole forest, visualize this (work in progress)
5. Train ML on the server side (not working yet)


Todo:
1. Add NDVI to feature space
2. Make it easier to select time frame and sampling method
3. Load dNBR and filter satellite image to burned pixels before predicting a value for each pixel

FR's comments:
- Add other next steps: 1) Migrate to EOPF, 2) Build UDF for ML and forest prediction
- Group imports once at the top of the notebook
- In general, can you combine function helpers into two cells: 1) For feature classification, 2) For forest prediction. We need to merge some functions. 
- After defining function helpers, you can do the pipeline in different sections: 1) Feature classification, 2) Forest prediction. This helps structuring the notebook and steering focus to outputs
- Decide whether you want to run the pipeline with `main()` or run each function one by one. The pipeline for feature classification is run with `main()` and for forest prediction is created by running the function one by one
- We might need to move this to dNBR notebook

In [1]:
## Remove unused libraries

import matplotlib.pyplot as plt
from PIL import Image
import openeo
from openeo.api.process import Parameter
from openeo_udp import ParameterManager
import geopandas
import json
import logging
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

logging.basicConfig(level=logging.INFO)

## Setup connection to OpenEO

In [2]:
## I commented out the following cell, you can review and delete it after
## This is because I moved your parameter definition to machine_learning.params.py
## Then I import them programmatically in the following cells

In [3]:
# connection = openeo.connect(
#     url="https://openeo.dataspace.copernicus.eu/"
# ).authenticate_oidc()

# time_frame = ["2024-01-01","2024-12-30"]
# time_post= ["2024-09-30", "2024-10-10"] 
# time_pre=  ["2024-08-20", "2024-09-01"]  

# current_params = {
#         "location_name": "Reriz e Gafanhao, North Portugal",
#         "bounding_box": Parameter(
#             "bounding_box",
#             description="Fire area in 2024",
#             default= {"west": -8.23, "south": 40.76, "east": -7.78, "north": 41}
# ),
#         "time": Parameter(
#             "time",
#             description="Temporal range for data acquisition",
#             # default=["2024-09-30", "2024-10-30"], # + last
#             default=["2024-08-01", "2024-09-01"],  # + last
#         ),
#         "bands": Parameter(
#             "bands",
#             description="Sentinel-2 bands required for APA calculation",
#             default=  ["B02","B03","B04","B05","B07","B08","B8A","B11","B12"]
# #,
#         ),
#         "collection": Parameter(
#             "collection",
#             description="Data collection identifier",
#             default="SENTINEL2_L2A",
#         ),
#         "cloud_cover": Parameter(
#             "cloud_cover",
#             description="Maximum cloud cover percentage",
#             default=30,
#         ),
#     }


In [4]:
# Initialize parameter manager
param_manager = ParameterManager('machine_learning.params.py')

# Display available options using the built-in helper
param_manager.print_options("Machine learning algorithm")

Available parameter sets for Machine learning algorithm:
  1. reriz_gafanhao_north_portugal_2024: Reriz e Gafanhao, North Portugal

Available OpenEO endpoints:
  1. localhost_dev: http://localhost:8082/
  2. ds_development: https://openeo.ds.io/
  3. copernicus_dataspace: https://openeo.dataspace.copernicus.eu/
  4. eopf_explorer: https://api.explorer.eopf.copernicus.eu/openeo

💡 Tip: Use param_manager.interactive_parameter_selection() for interactive selection,
or param_manager.quick_connect('set_name', 'endpoint') for direct connection.
To change selections, use the interactive widgets in the next cell.


In [5]:
# Connect using the a parameter set for a specified location on the Copernicus Data Space endpoint
connection, current_params = param_manager.quick_connect(
    param_set="reriz_gafanhao_north_portugal_2024",
    endpoint="copernicus_dataspace", # Connecting to CDSE
)

INFO:openeo.config:Loaded openEO client config from sources: []


🔄 Connecting to copernicus_dataspace...
📍 Using parameter set: reriz_gafanhao_north_portugal_2024


INFO:openeo.rest.connection:Found OIDC providers: ['CDSE']
INFO:openeo.rest.connection:No OIDC provider given, but only one available: 'CDSE'. Using that one.
INFO:openeo.rest.connection:Using default client_id 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e' from OIDC provider 'CDSE' info.
INFO:openeo.rest.connection:Found refresh token: trying refresh token based authentication.
INFO:openeo.rest.auth.oidc:Doing 'refresh_token' token request 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token' with post data fields ['grant_type', 'client_id', 'refresh_token'] (client_id 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e')
INFO:openeo.rest.connection:Obtained tokens: ['token_type', 'access_token', 'expires_in', 'id_token', 'refresh_token', 'scope']
INFO:openeo.rest.auth.config:Storing refresh token for issuer 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE' (client 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e')


Authenticated using refresh token.
✅ Successfully connected to copernicus_dataspace
✅ Parameters loaded and mapped for: Reriz e Gafanhao, North Portugal
🔄 Parameters mapped for endpoint copernicus_dataspace:


In [6]:
def load_and_sample(params, time, res=100):
    """
    Input: Set of parameters, time frame, desired resolution
    Output: Cube that contains image
    """ 
    s2cube = connection.load_collection(
    params["collection"].default,
    temporal_extent=params[time].default,
    spatial_extent=params["bounding_box"].default,
    bands=params["bands"].default,
    properties={
        "eo:cloud_cover": lambda x: x <= params["cloud_cover"].default,
    },
    )
    s2cube = s2cube.aggregate_temporal_period('month', reducer = 'median')
    # s2cube = s2cube.reduce_dimension(dimension="t", reducer="last")
    s2cube = s2cube.resample_spatial(
        resolution=[res, res],
      method="near",
    )

    return s2cube


# def calculate_ndvi(data):
#      """ 
#      Input: Cube with bands 04 and 8
#      Output: NBR
#      """
#      B04, B08 = (
#           data[2],
#           data[5]
#           )
#      ndvi = (B08 - B04) / (B08 + B04)
#      return ndvi

## Define ML pipeline that takes label path and data cube as inputs

FR's comment: it seems that `label_feature_matching` and `flatten_raw` do the same thing. Can you decide which one to use and make a more generic descriptive function name?

In [7]:
## I merged the label data importing with process_labels and added logging INFO just in case
## If this is okay, go ahead and delete load_labels() above
## If not, give explanation as to why you want to keep it separate
## If you don't think logging INFO is useful, you can delete it


# def load_labels(path):
#     """Load the labels from the geopackage and return them as a GeoDataFrame."""
#     # Load the labels from the geopackage
#     labels = geopandas.read_file(path, layer = "nfi")
#     return labels


def process_labels(path):
    """Process the labels and return them as a GeoJSON object."""
    labels = geopandas.read_file(path, layer="nfi")
    logging.info("Loaded %d labels from %s\n%s", len(labels), path, labels.head())

    # Convert to Coordinate System of S2 in Portugal
    labels = labels.to_crs("EPSG:4326")  
    
    # Drop "Other Forest" and "Non forest" labels
    labels = labels[labels["label"]!="Other Forest"]
    labels = labels[labels["label"]!="Non forest"]

    # Ensure right geometry type of labels
    labels["geometry"] = labels["geometry"].apply(lambda g: g.geoms[0] if g.geom_type == "MultiPoint" else g)
    
    # Convert to Geojson
    geojson = json.loads(labels[["label", "geometry"]].to_json())

    return labels, geojson


def load_features(s2cube, geojson):
    """Aggregate satellite data spatially over label points and return an openEO process node."""
    features = s2cube.aggregate_spatial(geojson, reducer="mean")
    return features


def download_features(features):
    """Download the features as a JSON file."""
    job = features.create_job(out_format="JSON")
    job.start_and_wait()
    job.get_results().download_files("features_output")
    with open("features_output/timeseries.json") as f:
        raw = json.load(f)
    return raw


## FR: Merge this with flatten_raw
def label_feature_matching(raw, nfi_labels):
    timestamps = list(raw.keys())
    n_points = len(nfi_labels)
    X = []
    for i in range(n_points):
        row_features = []
        for ts in timestamps:
            row_features.extend(raw[ts][i])
        X.append(row_features)
    X = np.array(X, dtype=float)
    y = nfi_labels["label"].values
    return X,y


def ml(X,y):
    X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    )
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    return clf, classification_report(y_test, y_pred)


def main(path, cube):
    labels, geojson = process_labels(path)
    features = load_features(cube, geojson)
    raw = download_features(features)
    X,y = label_feature_matching(raw, labels)
    clf, report = ml(X,y)
    return clf, report, raw

## Train ML algorithm
Define time frame, resolution and sample the image, then run the main that returns the machine learning algorithm and its performance measured on the test set

In [8]:
# Define Resolution and timeframe 
res = 20

# `time` argument can be filled by "time_post" or "time_pre"
s2cube = load_and_sample(current_params, "time_post", res)

# ML on tree classification given the label data nfi_labels.gpkg
classifier, report, raw = main("nfi_labels.gpkg", s2cube)

INFO:root:Loaded 561 labels from nfi_labels.gpkg
        label                               geometry
0  Eucalyptus  MULTIPOINT ((566705.671 4534428.416))
1  Eucalyptus  MULTIPOINT ((567200.516 4534933.176))
2  Eucalyptus  MULTIPOINT ((567710.234 4533938.527))
3  Eucalyptus  MULTIPOINT ((567705.277 4534438.331))
4  Eucalyptus  MULTIPOINT ((568204.981 4534443.287))


0:00:00 Job 'j-26062908374743708d72bda595cdf64e': send 'start'
0:00:04 Job 'j-26062908374743708d72bda595cdf64e': queued (progress 0%)
0:00:09 Job 'j-26062908374743708d72bda595cdf64e': queued (progress 0%)
0:00:16 Job 'j-26062908374743708d72bda595cdf64e': queued (progress 0%)
0:00:25 Job 'j-26062908374743708d72bda595cdf64e': running (progress N/A)
0:00:35 Job 'j-26062908374743708d72bda595cdf64e': running (progress N/A)
0:00:47 Job 'j-26062908374743708d72bda595cdf64e': running (progress N/A)
0:01:03 Job 'j-26062908374743708d72bda595cdf64e': running (progress N/A)
0:01:22 Job 'j-26062908374743708d72bda595cdf64e': running (progress N/A)
0:01:47 Job 'j-26062908374743708d72bda595cdf64e': finished (progress 100%)


INFO:openeo.rest.job:Downloading Job result asset 'timeseries.json' from https://s3.waw3-1.openeo.v1.dataspace.copernicus.eu/openeo-data-prod-waw4-1/batch_jobs/j-26062908374743708d72bda595cdf64e/timeseries.json?X-Proxy-Head-As-Get=true&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=654ba1d29d6c42c087da6fc0bbde43f8%2F20260629%2Fwaw4-1%2Fs3%2Faws4_request&X-Amz-Date=20260629T083935Z&X-Amz-Expires=86400&X-Amz-SignedHeaders=host&X-Amz-Security-Token=eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJyb2xlX2FybiI6ImFybjpvcGVuZW93czppYW06Ojpyb2xlL29wZW5lby1kYXRhLXByb2Qtd2F3NC0xLXdvcmtzcGFjZSIsImluaXRpYWxfaXNzdWVyIjoib3BlbmVvLnByb2Qud2F3My0xLm9wZW5lby1pbnQudjEuZGF0YXNwYWNlLmNvcGVybmljdXMuZXUiLCJodHRwczovL2F3cy5hbWF6b24uY29tL3RhZ3MiOnsicHJpbmNpcGFsX3RhZ3MiOnsiam9iX2lkIjpbImotMjYwNjI5MDgzNzQ3NDM3MDhkNzJiZGE1OTVjZGY2NGUiXSwidXNlcl9pZCI6WyI5NGM4MGZmZS0xYTUzLTQ2YTgtOGUyYy1mMGZhZjM2NGNmYzAiXX0sInRyYW5zaXRpdmVfdGFnX2tleXMiOlsidXNlcl9pZCIsImpvYl9pZCJdfSwiaXNzIjoic3RzLndhdzMtMS5vcGVuZW8udjEuZGF0YXNwYWNlLmNvcG

In [9]:
print(report)

                precision    recall  f1-score   support

    Eucalyptus       0.58      0.39      0.47        18
Maritime Pines       0.63      0.69      0.66        32
     Shrubland       0.77      0.81      0.79        59

      accuracy                           0.71       109
     macro avg       0.66      0.63      0.64       109
  weighted avg       0.70      0.71      0.70       109



## Predict Values

- Idea: Predict a value for each pixel in the satellite image
- Pipeline: 
1. Load satellite image(s) on the entire area of the burned forest (in progress)
2. Apply ML algorithm to predict a value for each pixel 
3. Plot this

In [ ]:
## FR: Any reason as to not put these imports on the top of the notebook?
## I think it's better to keep these in one cell, we have to merge some processes
import glob
import json
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

## FR: can you combine this download_area_cube and download_features. 
## Just call it as download_batch_job (see the recommendation in the cell below)
def download_area_cube(cube, out_folder="area_output"):
    """Download the full-area cube (same multi-date cube used for training) as netCDF."""
    job = cube.create_job(out_format="netCDF")
    job.start_and_wait()
    job.get_results().download_files(out_folder)
    return out_folder


def load_area_cube(out_folder="area_output"):
    """Load the downloaded netCDF into an xarray Dataset."""
    nc_path = glob.glob(f"{out_folder}/*.nc")[0]
    ds = xr.open_dataset(nc_path)
    return ds


def build_feature_array(ds, band_order, time_order):
    """
    Stack bands x dates into a (n_features, height, width) array,
    using the SAME order as label_feature_matching: for each date, all bands.
    band_order / time_order must match what was used to build training X.
    """
    layers = []
    for t in time_order:
        for b in band_order:
            layers.append(ds[b].sel(t=t).values)
    arr = np.stack(layers, axis=0)  # (n_features, height, width)
    return arr


def predict_area(clf, arr):
    """Reshape to (n_pixels, n_features), predict, reshape back to (height, width)."""
    n_features, height, width = arr.shape
    X_pixels = arr.reshape(n_features, height * width).T

    valid_mask = ~np.isnan(X_pixels).any(axis=1)
    preds = np.full(X_pixels.shape[0], fill_value="NoData", dtype=object)
    preds[valid_mask] = clf.predict(X_pixels[valid_mask])

    return preds.reshape(height, width)


def save_predictions(pred_map, path="predictions.npy"):
    """Store the predicted class map for later reuse."""
    np.save(path, pred_map)


def plot_classification(pred_map):
    """Plot the classified raster with a discrete legend."""
    unique_classes = sorted(set(pred_map.flatten()) - {"NoData"})
    class_to_int = {cls: i for i, cls in enumerate(unique_classes)}
    class_to_int["NoData"] = -1

    int_map = np.vectorize(class_to_int.get)(pred_map)

    cmap = plt.get_cmap("tab10", len(unique_classes))
    colors = ["lightgrey"] + [cmap(i) for i in range(len(unique_classes))]
    full_cmap = ListedColormap(colors)
    bounds = list(range(-1, len(unique_classes) + 1))
    norm = BoundaryNorm(bounds, full_cmap.N)

    fig, ax = plt.subplots(figsize=(10, 10))
    im = ax.imshow(int_map, cmap=full_cmap, norm=norm)
    cbar = plt.colorbar(im, ax=ax, ticks=list(class_to_int.values()))
    cbar.ax.set_yticklabels(list(class_to_int.keys()))
    ax.set_title("Predicted Land Cover")
    ax.set_axis_off()
    plt.show()

## Can you separate the cells for functions and processes? 
## Do you want to run the processes with main() or one by one?
## For feature classification, you ran the pipeline with main() and for prediction you run it one-by-one

folder = download_area_cube(s2cube)        # same multi-date cube used in load_features() earlier
ds = load_area_cube(folder)

# These MUST match what was used to build training X — check raw.keys() order
# and the band list passed to load_collection for the original cube.
band_order = ["B02","B03","B04","B05","B07","B08","B8A","B11","B12"]   # e.g. ["B02","B03","B04","B05","B07","B08","B8A","B11","B12"]
time_order = list(raw.keys())  # reuse the exact timestamp order from training

arr = build_feature_array(ds, band_order, time_order)
print(arr.shape)  # sanity check: should be (18, height, width)

pred_map = predict_area(classifier, arr)
save_predictions(pred_map)
plot_classification(pred_map)

0:00:00 Job 'j-26062510312843f7acdf736e72260d84': send 'start'
0:00:03 Job 'j-26062510312843f7acdf736e72260d84': created (progress 0%)
0:00:08 Job 'j-26062510312843f7acdf736e72260d84': queued (progress 0%)
0:00:14 Job 'j-26062510312843f7acdf736e72260d84': queued (progress 0%)
0:00:22 Job 'j-26062510312843f7acdf736e72260d84': queued (progress 0%)
0:00:32 Job 'j-26062510312843f7acdf736e72260d84': queued (progress 0%)
0:00:45 Job 'j-26062510312843f7acdf736e72260d84': queued (progress 0%)
0:01:00 Job 'j-26062510312843f7acdf736e72260d84': queued (progress 0%)
0:01:20 Job 'j-26062510312843f7acdf736e72260d84': running (progress N/A)
0:01:44 Job 'j-26062510312843f7acdf736e72260d84': running (progress N/A)
0:02:14 Job 'j-26062510312843f7acdf736e72260d84': finished (progress 100%)


ValueError: found the following matches with the input file in xarray's IO backends: ['netcdf4', 'h5netcdf']. But their dependencies may not be installed, see:
https://docs.xarray.dev/en/stable/user-guide/io.html 
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html

# Machine Learning using Sentinel 2 images through OpenEO and Forest Inventory data

In this document the Sentinel 2 images that cover the burned area after a forest fire are downloaded and a Machine Learning algorithm is trained to classify the vegetation species before the fire.

1. Connect to openEO and define the Cube
2. Define ML pipeline
3. Train ML algorithm locally
4. Predict values for the whole forest, visualize this (work in progress)
5. Train ML on the server side (not working yet)


Todo:
1. Add NDVI to feature space
2. Make it easier to select time frame and sampling method
3. Load dNBR and filter satellite image to burned pixels before predicting a value for each pixel

FR's comments:
- Add other next steps: 1) Migrate to EOPF, 2) Build UDF for ML and forest prediction

In [ ]:
## Remove unused libraries

import matplotlib.pyplot as plt
from PIL import Image
import openeo
from openeo.api.process import Parameter
from openeo_udp import ParameterManager
import geopandas
import json
import logging
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

logging.basicConfig(level=logging.INFO)

## Setup connection to OpenEO

In [ ]:
# connection = openeo.connect(
#     url="https://openeo.dataspace.copernicus.eu/"
# ).authenticate_oidc()

# time_frame = ["2024-01-01","2024-12-30"]
# time_post= ["2024-09-30", "2024-10-10"] 
# time_pre=  ["2024-08-20", "2024-09-01"]  

# current_params = {
#         "location_name": "Reriz e Gafanhao, North Portugal",
#         "bounding_box": Parameter(
#             "bounding_box",
#             description="Fire area in 2024",
#             default= {"west": -8.23, "south": 40.76, "east": -7.78, "north": 41}
# ),
#         "time": Parameter(
#             "time",
#             description="Temporal range for data acquisition",
#             # default=["2024-09-30", "2024-10-30"], # + last
#             default=["2024-08-01", "2024-09-01"],  # + last
#         ),
#         "bands": Parameter(
#             "bands",
#             description="Sentinel-2 bands required for APA calculation",
#             default=  ["B02","B03","B04","B05","B07","B08","B8A","B11","B12"]
# #,
#         ),
#         "collection": Parameter(
#             "collection",
#             description="Data collection identifier",
#             default="SENTINEL2_L2A",
#         ),
#         "cloud_cover": Parameter(
#             "cloud_cover",
#             description="Maximum cloud cover percentage",
#             default=30,
#         ),
#     }


In [ ]:
def load_and_sample(params, time, res=100):
    """
    Input: Set of parameters, time frame, desired resolution
    Output: Cube that contains image
    """ 
    s2cube = connection.load_collection(
    params["collection"].default,
    temporal_extent=params[time].default,
    spatial_extent=params["bounding_box"].default,
    bands=params["bands"].default,
    properties={
        "eo:cloud_cover": lambda x: x <= params["cloud_cover"].default,
    },
    )
    s2cube = s2cube.aggregate_temporal_period('month', reducer = 'median')
    # s2cube = s2cube.reduce_dimension(dimension="t", reducer="last")
    s2cube = s2cube.resample_spatial(
        resolution=[res, res],
      method="near",
    )

    return s2cube


# def calculate_ndvi(data):
#      """ 
#      Input: Cube with bands 04 and 8
#      Output: NBR
#      """
#      B04, B08 = (
#           data[2],
#           data[5]
#           )
#      ndvi = (B08 - B04) / (B08 + B04)
#      return ndvi

## Define ML pipeline that takes label path and data cube as inputs

FR's comment: it seems that `label_feature_matching` and `flatten_raw` do the same thing. Can you decide which one to use and make a more generic descriptive function name?

In [ ]:
## I merged the label data importing with process_labels and added logging INFO just in case
## If this is okay, go ahead and delete load_labels() above
## If not, give explanation as to why you want to keep it separate
## If you don't think logging INFO is useful, you can delete it


# def load_labels(path):
#     """Load the labels from the geopackage and return them as a GeoDataFrame."""
#     # Load the labels from the geopackage
#     labels = geopandas.read_file(path, layer = "nfi")
#     return labels


def process_labels(path):
    """Process the labels and return them as a GeoJSON object."""
    labels = geopandas.read_file(path, layer="nfi")
    logging.info("Loaded %d labels from %s\n%s", len(labels), path, labels.head())

    # Convert to Coordinate System of S2 in Portugal
    labels = labels.to_crs("EPSG:4326")  
    
    # Drop "Other Forest" and "Non forest" labels
    labels = labels[labels["label"]!="Other Forest"]
    labels = labels[labels["label"]!="Non forest"]

    # Ensure right geometry type of labels
    labels["geometry"] = labels["geometry"].apply(lambda g: g.geoms[0] if g.geom_type == "MultiPoint" else g)
    
    # Convert to Geojson
    geojson = json.loads(labels[["label", "geometry"]].to_json())

    return labels, geojson


def load_features(s2cube, geojson):
    """Aggregate satellite data spatially over label points and return an openEO process node."""
    features = s2cube.aggregate_spatial(geojson, reducer="mean")
    return features


def download_features(features):
    """Download the features as a JSON file."""
    job = features.create_job(out_format="JSON")
    job.start_and_wait()
    job.get_results().download_files("features_output")
    with open("features_output/timeseries.json") as f:
        raw = json.load(f)
    return raw


## FR: Merge this with flatten_raw
def label_feature_matching(raw, nfi_labels):
    timestamps = list(raw.keys())
    n_points = len(nfi_labels)
    X = []
    for i in range(n_points):
        row_features = []
        for ts in timestamps:
            row_features.extend(raw[ts][i])
        X.append(row_features)
    X = np.array(X, dtype=float)
    y = nfi_labels["label"].values
    return X,y


def ml(X,y):
    X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    )
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    return clf, classification_report(y_test, y_pred)


def main(path, cube):
    labels, geojson = process_labels(path)
    features = load_features(cube, geojson)
    raw = download_features(features)
    X,y = label_feature_matching(raw, labels)
    clf, report = ml(X,y)
    return clf, report, raw

## Train ML algorithm
Define time frame, resolution and sample the image, then run the main that returns the machine learning algorithm and its performance measured on the test set

In [ ]:
# Define Resolution and timeframe 
res = 20

# `time` argument can be filled by "time_post" or "time_pre"
s2cube = load_and_sample(current_params, "time_post", res)

# ML on tree classification given the label data nfi_labels.gpkg
classifier, report, raw = main("nfi_labels.gpkg", s2cube)

INFO:root:Loaded 561 labels from nfi_labels.gpkg
        label                               geometry
0  Eucalyptus  MULTIPOINT ((566705.671 4534428.416))
1  Eucalyptus  MULTIPOINT ((567200.516 4534933.176))
2  Eucalyptus  MULTIPOINT ((567710.234 4533938.527))
3  Eucalyptus  MULTIPOINT ((567705.277 4534438.331))
4  Eucalyptus  MULTIPOINT ((568204.981 4534443.287))


0:00:00 Job 'j-26062908374743708d72bda595cdf64e': send 'start'
0:00:04 Job 'j-26062908374743708d72bda595cdf64e': queued (progress 0%)
0:00:09 Job 'j-26062908374743708d72bda595cdf64e': queued (progress 0%)
0:00:16 Job 'j-26062908374743708d72bda595cdf64e': queued (progress 0%)
0:00:25 Job 'j-26062908374743708d72bda595cdf64e': running (progress N/A)
0:00:35 Job 'j-26062908374743708d72bda595cdf64e': running (progress N/A)
0:00:47 Job 'j-26062908374743708d72bda595cdf64e': running (progress N/A)
0:01:03 Job 'j-26062908374743708d72bda595cdf64e': running (progress N/A)
0:01:22 Job 'j-26062908374743708d72bda595cdf64e': running (progress N/A)
0:01:47 Job 'j-26062908374743708d72bda595cdf64e': finished (progress 100%)


INFO:openeo.rest.job:Downloading Job result asset 'timeseries.json' from https://s3.waw3-1.openeo.v1.dataspace.copernicus.eu/openeo-data-prod-waw4-1/batch_jobs/j-26062908374743708d72bda595cdf64e/timeseries.json?X-Proxy-Head-As-Get=true&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=654ba1d29d6c42c087da6fc0bbde43f8%2F20260629%2Fwaw4-1%2Fs3%2Faws4_request&X-Amz-Date=20260629T083935Z&X-Amz-Expires=86400&X-Amz-SignedHeaders=host&X-Amz-Security-Token=eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJyb2xlX2FybiI6ImFybjpvcGVuZW93czppYW06Ojpyb2xlL29wZW5lby1kYXRhLXByb2Qtd2F3NC0xLXdvcmtzcGFjZSIsImluaXRpYWxfaXNzdWVyIjoib3BlbmVvLnByb2Qud2F3My0xLm9wZW5lby1pbnQudjEuZGF0YXNwYWNlLmNvcGVybmljdXMuZXUiLCJodHRwczovL2F3cy5hbWF6b24uY29tL3RhZ3MiOnsicHJpbmNpcGFsX3RhZ3MiOnsiam9iX2lkIjpbImotMjYwNjI5MDgzNzQ3NDM3MDhkNzJiZGE1OTVjZGY2NGUiXSwidXNlcl9pZCI6WyI5NGM4MGZmZS0xYTUzLTQ2YTgtOGUyYy1mMGZhZjM2NGNmYzAiXX0sInRyYW5zaXRpdmVfdGFnX2tleXMiOlsidXNlcl9pZCIsImpvYl9pZCJdfSwiaXNzIjoic3RzLndhdzMtMS5vcGVuZW8udjEuZGF0YXNwYWNlLmNvcG

0:00:00 Job 'j-2606251027144b3ca5c0d6788905cbf6': send 'start'
0:00:03 Job 'j-2606251027144b3ca5c0d6788905cbf6': queued (progress 0%)
0:00:09 Job 'j-2606251027144b3ca5c0d6788905cbf6': queued (progress 0%)
0:00:16 Job 'j-2606251027144b3ca5c0d6788905cbf6': queued (progress 0%)
0:00:25 Job 'j-2606251027144b3ca5c0d6788905cbf6': running (progress N/A)
0:00:35 Job 'j-2606251027144b3ca5c0d6788905cbf6': running (progress N/A)
0:00:48 Job 'j-2606251027144b3ca5c0d6788905cbf6': running (progress N/A)
0:01:04 Job 'j-2606251027144b3ca5c0d6788905cbf6': running (progress N/A)
0:01:24 Job 'j-2606251027144b3ca5c0d6788905cbf6': running (progress N/A)
0:01:48 Job 'j-2606251027144b3ca5c0d6788905cbf6': running (progress N/A)
0:02:19 Job 'j-2606251027144b3ca5c0d6788905cbf6': finished (progress 100%)


In [ ]:
print(report)

                precision    recall  f1-score   support

    Eucalyptus       0.64      0.39      0.48        18
Maritime Pines       0.66      0.72      0.69        32
     Shrubland       0.76      0.81      0.79        59

      accuracy                           0.72       109
     macro avg       0.69      0.64      0.65       109
  weighted avg       0.71      0.72      0.71       109



## Predict Values

- Idea: Predict a value for each pixel in the satellite image
- Pipeline: 
1. Load satellite image(s) on the entire area of the burned forest (in progress)
2. Apply ML algorithm to predict a value for each pixel 
3. Plot this

In [ ]:
## FR: Any reason as to not put these imports on the top of the notebook?
## I think it's better to keep these in one cell, we have to merge some processes
import glob
import json
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

## FR: can you combine this download_area_cube and download_features. 
## Just call it as download_batch_job (see the recommendation in the cell below)
def download_area_cube(cube, out_folder="area_output"):
    """Download the full-area cube (same multi-date cube used for training) as netCDF."""
    job = cube.create_job(out_format="netCDF")
    job.start_and_wait()
    job.get_results().download_files(out_folder)
    return out_folder


def load_area_cube(out_folder="area_output"):
    """Load the downloaded netCDF into an xarray Dataset."""
    nc_path = glob.glob(f"{out_folder}/*.nc")[0]
    ds = xr.open_dataset(nc_path)
    return ds


def build_feature_array(ds, band_order, time_order):
    """
    Stack bands x dates into a (n_features, height, width) array,
    using the SAME order as label_feature_matching: for each date, all bands.
    band_order / time_order must match what was used to build training X.
    """
    layers = []
    for t in time_order:
        for b in band_order:
            layers.append(ds[b].sel(t=t).values)
    arr = np.stack(layers, axis=0)  # (n_features, height, width)
    return arr


def predict_area(clf, arr):
    """Reshape to (n_pixels, n_features), predict, reshape back to (height, width)."""
    n_features, height, width = arr.shape
    X_pixels = arr.reshape(n_features, height * width).T

    valid_mask = ~np.isnan(X_pixels).any(axis=1)
    preds = np.full(X_pixels.shape[0], fill_value="NoData", dtype=object)
    preds[valid_mask] = clf.predict(X_pixels[valid_mask])

    return preds.reshape(height, width)


def save_predictions(pred_map, path="predictions.npy"):
    """Store the predicted class map for later reuse."""
    np.save(path, pred_map)


def plot_classification(pred_map):
    """Plot the classified raster with a discrete legend."""
    unique_classes = sorted(set(pred_map.flatten()) - {"NoData"})
    class_to_int = {cls: i for i, cls in enumerate(unique_classes)}
    class_to_int["NoData"] = -1

    int_map = np.vectorize(class_to_int.get)(pred_map)

    cmap = plt.get_cmap("tab10", len(unique_classes))
    colors = ["lightgrey"] + [cmap(i) for i in range(len(unique_classes))]
    full_cmap = ListedColormap(colors)
    bounds = list(range(-1, len(unique_classes) + 1))
    norm = BoundaryNorm(bounds, full_cmap.N)

    fig, ax = plt.subplots(figsize=(10, 10))
    im = ax.imshow(int_map, cmap=full_cmap, norm=norm)
    cbar = plt.colorbar(im, ax=ax, ticks=list(class_to_int.values()))
    cbar.ax.set_yticklabels(list(class_to_int.keys()))
    ax.set_title("Predicted Land Cover")
    ax.set_axis_off()
    plt.show()

## Can you separate the cells for functions and processes? 

folder = download_area_cube(s2cube)        # same multi-date cube used in load_features() earlier
ds = load_area_cube(folder)

# These MUST match what was used to build training X — check raw.keys() order
# and the band list passed to load_collection for the original cube.
band_order = ["B02","B03","B04","B05","B07","B08","B8A","B11","B12"]   # e.g. ["B02","B03","B04","B05","B07","B08","B8A","B11","B12"]
time_order = list(raw.keys())  # reuse the exact timestamp order from training

arr = build_feature_array(ds, band_order, time_order)
print(arr.shape)  # sanity check: should be (18, height, width)

pred_map = predict_area(classifier, arr)
save_predictions(pred_map)
plot_classification(pred_map)

0:00:00 Job 'j-26062510312843f7acdf736e72260d84': send 'start'
0:00:03 Job 'j-26062510312843f7acdf736e72260d84': created (progress 0%)
0:00:08 Job 'j-26062510312843f7acdf736e72260d84': queued (progress 0%)
0:00:14 Job 'j-26062510312843f7acdf736e72260d84': queued (progress 0%)
0:00:22 Job 'j-26062510312843f7acdf736e72260d84': queued (progress 0%)
0:00:32 Job 'j-26062510312843f7acdf736e72260d84': queued (progress 0%)
0:00:45 Job 'j-26062510312843f7acdf736e72260d84': queued (progress 0%)
0:01:00 Job 'j-26062510312843f7acdf736e72260d84': queued (progress 0%)
0:01:20 Job 'j-26062510312843f7acdf736e72260d84': running (progress N/A)
0:01:44 Job 'j-26062510312843f7acdf736e72260d84': running (progress N/A)
0:02:14 Job 'j-26062510312843f7acdf736e72260d84': finished (progress 100%)


ValueError: found the following matches with the input file in xarray's IO backends: ['netcdf4', 'h5netcdf']. But their dependencies may not be installed, see:
https://docs.xarray.dev/en/stable/user-guide/io.html 
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html

In [ ]:
import geopandas as gpd
from shapely.geometry import Point


spatial_extent = {"west": -8.23, "south": 40.76, "east": -7.78, "north": 41}


def make_grid(spatial_extent, spacing=20):
    """Grid of points covering the cube's bounding box, `spacing` meters apart."""
    bbox = gpd.GeoDataFrame(
        geometry=[box(spatial_extent["west"], spatial_extent["south"],
                      spatial_extent["east"], spatial_extent["north"])],
        crs="EPSG:4326"
    ).to_crs("EPSG:32629")  # project's working CRS (UTM 29N, matches labels)

    minx, miny, maxx, maxy = bbox.total_bounds
    xs = np.arange(minx, maxx, spacing)
    ys = np.arange(miny, maxy, spacing)
    points = [Point(x, y) for x in xs for y in ys]

    grid = gpd.GeoDataFrame(geometry=points, crs="EPSG:32629")
    return grid.to_crs("EPSG:4326")  # back to WGS84 for GeoJSON

from shapely.geometry import box

grid = make_grid(spatial_extent, spacing=20)  # spatial_extent = the dict you used in load_collection

In [ ]:
def flatten_raw(raw, n_points):
    """Flatten raw timeseries dict into a feature matrix, one row per point."""
    timestamps = list(raw.keys())
    X = [[v for ts in timestamps for v in raw[ts][i]] for i in range(n_points)]
    return np.array(X, dtype=float)


grid_geojson = json.loads(grid.to_json())
features = load_features(s2cube, grid_geojson)     # unchanged
raw = download_features(features)                # unchanged

X_grid = flatten_raw(raw, len(grid))              # same flattening as label_feature_matching
grid["pred"] = classifier.predict(X_grid)

KeyboardInterrupt: 

In [ ]:
print(len(grid))

2587494


## Online Training

In [ ]:
cube = s2cube
path = "nfi_labels.gpkg"

labels = load_labels(path)
labels, geojson = process_labels(labels)
# features = load_features(cube, geojson)
# raw = download_features(features)
# X,y = label_feature_matching(raw, labels)

# X_train, X_test, y_train, y_test = train_test_split(
# X, y, test_size=0.2, random_state=42, stratify=y)
y_train, y_test = train_test_split(labels, test_size=0.2, random_state=42)


In [ ]:
cube = cube.reduce_dimension(dimension="t", reducer="last")
X_train = cube.aggregate_spatial(json.loads(y_train.to_json()), reducer="mean")
clf = X_train.fit_class_random_forest(target=json.loads(y_train.to_json()), num_trees=200)
model = clf.save_ml_model()

In [ ]:
job_options = {
    "executor-memory": "4G",
    "python-memory": "2500m",
}

training_job = model.create_job(
    title="Training-job Dynamic LC", job_options=job_options
)
training_job.start_and_wait()

0:00:00 Job 'j-26062509385348fc8555ab628f18d251': send 'start'
0:00:05 Job 'j-26062509385348fc8555ab628f18d251': queued (progress 0%)
0:00:11 Job 'j-26062509385348fc8555ab628f18d251': queued (progress 0%)
0:00:18 Job 'j-26062509385348fc8555ab628f18d251': queued (progress 0%)
0:00:27 Job 'j-26062509385348fc8555ab628f18d251': queued (progress 0%)
0:00:38 Job 'j-26062509385348fc8555ab628f18d251': running (progress N/A)
0:00:51 Job 'j-26062509385348fc8555ab628f18d251': running (progress N/A)
0:01:07 Job 'j-26062509385348fc8555ab628f18d251': running (progress N/A)
0:01:28 Job 'j-26062509385348fc8555ab628f18d251': running (progress N/A)
0:01:52 Job 'j-26062509385348fc8555ab628f18d251': running (progress N/A)
0:02:23 Job 'j-26062509385348fc8555ab628f18d251': running (progress N/A)
0:03:02 Job 'j-26062509385348fc8555ab628f18d251': error (progress N/A)
Your batch job 'j-26062509385348fc8555ab628f18d251' failed. Error logs:
[]
Full logs can be inspected in an openEO (web) editor or with `connect

JobFailedException: Batch job 'j-26062509385348fc8555ab628f18d251' didn't finish successfully. Status: error (after 0:03:03).

In [ ]:
connection.job('j-26062418124140408ffb3a6c20cfa019').logs()

[{'id': '[1782324788291, 117647]',
  'time': '2026-06-24T18:13:08.291Z',
  'level': 'info',
  'message': "batch_job.py sys.version='3.11.7 (main, Oct  9 2024, 00:00:00) [GCC 11.4.1 20231218 (Red Hat 11.4.1-3)]'"},
 {'id': '[1782324788291, 667958]',
  'time': '2026-06-24T18:13:08.291Z',
  'level': 'info',
  'message': 'Starting batch job os.getpid()=84: start 2026-06-24 18:13:08.290825'},
 {'id': '[1782324788830, 255303]',
  'time': '2026-06-24T18:13:08.830Z',
  'level': 'info',
  'message': 'Loading custom processes from /opt/openeo-geopyspark-k8s-custom-processes/src/openeo_geopyspark_k8s_custom_processes/custom_processes.py'},
 {'id': '[1782324788832, 281498]',
  'time': '2026-06-24T18:13:08.832Z',
  'level': 'info',
  'message': 'Overriding process sar_backscatter (namespace backend)'},
 {'id': '[1782324788832, 307622]',
  'time': '2026-06-24T18:13:08.832Z',
  'level': 'info',
  'message': "load_custom_processes: exec loaded '/opt/openeo-geopyspark-k8s-custom-processes/src/openeo_geopyspark_k8s_custom_processes/custom_processes.py'"},
 {'id': '[1782324788832, 816723]',
  'time': '2026-06-24T18:13:08.832Z',
  'level': 'info',
  'message': 'Overriding process sar_backscatter (namespace backend)'},
 {'id': '[1782324803956, 685831]',
  'time': '2026-06-24T18:13:23.956Z',
  'level': 'info',
  'message': 'Job spec: {\n "process_graph": {\n  "loadcollection1": {\n   "process_id": "load_collection",\n   "arguments": {\n    "bands": [\n     "B02",\n     "B03",\n     "B04",\n     "B05",\n     "B07",\n     "B08",\n     "B8A",\n     "B11",\n     "B12"\n    ],\n    "id": "SENTINEL2_L2A",\n    "properties": {\n     "eo:cloud_cover": {\n      "process_graph": {\n       "lte1": {\n        "process_id": "lte",\n        "arguments": {\n         "x": {\n          "from_parameter": "value"\n         },\n         "y": 30\n        },\n        "result": true\n       }\n      }\n     }\n    },\n    "spatial_extent": {\n     "west": -8.23,\n     "south": 40.76,\n     "east": -7.78,\n     "north": 41\n    },\n    "temporal_extent": [\n     "2024-09-30",\n     "2024-10-10"\n    ]\n   }\n  },\n  "aggregatetemporalperiod1": {\n   "process_id": "aggregate_temporal_period",\n   "arguments": {\n    "data": {\n     "from_node": "loadcollection1"\n    },\n    "period": "month",\n    "reducer": {\n     "process_graph": {\n      "median1": {\n       "process_id": "median",\n       "arguments": {\n        "data": {\n         "from_parameter": "data"\n        }\n       },\n       "result": true\n      }\n     }\n    }\n   }\n  },\n  "resamplespatial1": {\n   "process_id": "resample_spatial",\n   "arguments": {\n    "align": "upper-left",\n    "data": {\n     "from_node": "aggregatetemporalperiod1"\n    },\n    "method": "near",\n    "projection": null,\n    "resolution": [\n     20,\n     20\n    ]\n   }\n  },\n  "reducedimension1": {\n   "process_id": "reduce_dimension",\n   "arguments": {\n    "data": {\n     "from_node": "resamplespatial1"\n    },\n    "dimension": "t",\n    "reducer": {\n     "process_graph": {\n      "last1": {\n       "process_id": "last",\n       "arguments": {\n        "data": {\n         "from_parameter": "data"\n        }\n       },\n       "result": true\n      }\n     }\n    }\n   }\n  },\n  "aggregatespatial1": {\n   "process_id": "aggregate_spatial",\n   "arguments": {\n    "data": {\n     "from_node": "reducedimension1"\n    },\n    "geometries": {\n     "type": "FeatureCollection",\n     "features": [\n      {\n       "id": "215",\n       "type": "Feature",\n       "properties": {\n        "label": "Maritime Pines"\n       },\n       "geometry": {\n        "type": "Point",\n        "coordinates": [\n         -7.964046085398656,\n         40.88164564962622\n        ]\n       }\n      },\n      {\n       "id": "189",\n       "type": "Feature",\n       "properties": {\n        "label": "Shrubland"\n       },\n       "geometry": {\n        "type": "Point",\n        "coordinates": [\n         -7.999503387802833,\n         40.94922610

0:00:00 Job 'j-2606251018364139910376c8be04390f': send 'start'
0:00:03 Job 'j-2606251018364139910376c8be04390f': queued (progress 0%)
0:00:08 Job 'j-2606251018364139910376c8be04390f': queued (progress 0%)
0:00:14 Job 'j-2606251018364139910376c8be04390f': queued (progress 0%)
0:00:22 Job 'j-2606251018364139910376c8be04390f': queued (progress 0%)
0:00:32 Job 'j-2606251018364139910376c8be04390f': queued (progress 0%)
0:00:45 Job 'j-2606251018364139910376c8be04390f': running (progress N/A)
0:01:00 Job 'j-2606251018364139910376c8be04390f': running (progress N/A)
0:01:19 Job 'j-2606251018364139910376c8be04390f': running (progress N/A)
0:01:43 Job 'j-2606251018364139910376c8be04390f': running (progress N/A)
0:02:13 Job 'j-2606251018364139910376c8be04390f': finished (progress 100%)


## Visualize labels

In [ ]:
def plot_labels(labels, save_path=None):
    """Plot NFI label points colored by class, with a basemap if contextily is available."""
    labels_wgs84 = labels.to_crs("EPSG:3857")  # web-mercator, needed for basemap tiles

    fig, ax = plt.subplots(figsize=(10, 10))

    # one color per class, plotted as separate layers (so legend works cleanly)
    classes = sorted(labels_wgs84["label"].unique())
    cmap = plt.get_cmap("tab10")
    for i, cls in enumerate(classes):
        subset = labels_wgs84[labels_wgs84["label"] == cls]
        subset.plot(ax=ax, color=cmap(i % 10), markersize=15, label=cls)

    ax.set_title(f"NFI label points (n={len(labels)})")
    ax.legend(title="Label", loc="upper right", fontsize=8)
    ax.set_axis_off()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    # quick class balance printout alongside the plot
    print(labels["label"].value_counts())

plot_labels(labels)

In [ ]:
# def download_batch_job(cube, out_format, out_folder):
#     """Submit a batch job, wait for completion, and download results to out_folder."""
#     job = cube.create_job(out_format=out_format)
#     job.start_and_wait()
#     job.get_results().download_files(out_folder)
#     return out_folder

In [41]:
import geopandas as gpd
from shapely.geometry import Point


spatial_extent = {"west": -8.23, "south": 40.76, "east": -7.78, "north": 41}


def make_grid(spatial_extent, spacing=20):
    """Grid of points covering the cube's bounding box, `spacing` meters apart."""
    bbox = gpd.GeoDataFrame(
        geometry=[box(spatial_extent["west"], spatial_extent["south"],
                      spatial_extent["east"], spatial_extent["north"])],
        crs="EPSG:4326"
    ).to_crs("EPSG:32629")  # project's working CRS (UTM 29N, matches labels)

    minx, miny, maxx, maxy = bbox.total_bounds
    xs = np.arange(minx, maxx, spacing)
    ys = np.arange(miny, maxy, spacing)
    points = [Point(x, y) for x in xs for y in ys]

    grid = gpd.GeoDataFrame(geometry=points, crs="EPSG:32629")
    return grid.to_crs("EPSG:4326")  # back to WGS84 for GeoJSON

from shapely.geometry import box

grid = make_grid(spatial_extent, spacing=20)  # spatial_extent = the dict you used in load_collection

In [ ]:
def flatten_raw(raw, n_points):
    """Flatten raw timeseries dict into a feature matrix, one row per point."""
    timestamps = list(raw.keys())
    X = [[v for ts in timestamps for v in raw[ts][i]] for i in range(n_points)]
    return np.array(X, dtype=float)


grid_geojson = json.loads(grid.to_json())
features = load_features(s2cube, grid_geojson)     # unchanged
raw = download_features(features)                # unchanged

X_grid = flatten_raw(raw, len(grid))              # same flattening as label_feature_matching
grid["pred"] = classifier.predict(X_grid)

KeyboardInterrupt: 

In [43]:
print(len(grid))

2587494


## Online Training

In [ ]:
cube = s2cube
path = "nfi_labels.gpkg"

labels = load_labels(path)
labels, geojson = process_labels(labels)
# features = load_features(cube, geojson)
# raw = download_features(features)
# X,y = label_feature_matching(raw, labels)

# X_train, X_test, y_train, y_test = train_test_split(
# X, y, test_size=0.2, random_state=42, stratify=y)
y_train, y_test = train_test_split(labels, test_size=0.2, random_state=42)


In [10]:
cube = cube.reduce_dimension(dimension="t", reducer="last")
X_train = cube.aggregate_spatial(json.loads(y_train.to_json()), reducer="mean")
clf = X_train.fit_class_random_forest(target=json.loads(y_train.to_json()), num_trees=200)
model = clf.save_ml_model()

In [11]:
job_options = {
    "executor-memory": "4G",
    "python-memory": "2500m",
}

training_job = model.create_job(
    title="Training-job Dynamic LC", job_options=job_options
)
training_job.start_and_wait()

0:00:00 Job 'j-26062509385348fc8555ab628f18d251': send 'start'
0:00:05 Job 'j-26062509385348fc8555ab628f18d251': queued (progress 0%)
0:00:11 Job 'j-26062509385348fc8555ab628f18d251': queued (progress 0%)
0:00:18 Job 'j-26062509385348fc8555ab628f18d251': queued (progress 0%)
0:00:27 Job 'j-26062509385348fc8555ab628f18d251': queued (progress 0%)
0:00:38 Job 'j-26062509385348fc8555ab628f18d251': running (progress N/A)
0:00:51 Job 'j-26062509385348fc8555ab628f18d251': running (progress N/A)
0:01:07 Job 'j-26062509385348fc8555ab628f18d251': running (progress N/A)
0:01:28 Job 'j-26062509385348fc8555ab628f18d251': running (progress N/A)
0:01:52 Job 'j-26062509385348fc8555ab628f18d251': running (progress N/A)
0:02:23 Job 'j-26062509385348fc8555ab628f18d251': running (progress N/A)
0:03:02 Job 'j-26062509385348fc8555ab628f18d251': error (progress N/A)
Your batch job 'j-26062509385348fc8555ab628f18d251' failed. Error logs:
[]
Full logs can be inspected in an openEO (web) editor or with `connect

JobFailedException: Batch job 'j-26062509385348fc8555ab628f18d251' didn't finish successfully. Status: error (after 0:03:03).

In [12]:
connection.job('j-26062418124140408ffb3a6c20cfa019').logs()

[{'id': '[1782324788291, 117647]',
  'time': '2026-06-24T18:13:08.291Z',
  'level': 'info',
  'message': "batch_job.py sys.version='3.11.7 (main, Oct  9 2024, 00:00:00) [GCC 11.4.1 20231218 (Red Hat 11.4.1-3)]'"},
 {'id': '[1782324788291, 667958]',
  'time': '2026-06-24T18:13:08.291Z',
  'level': 'info',
  'message': 'Starting batch job os.getpid()=84: start 2026-06-24 18:13:08.290825'},
 {'id': '[1782324788830, 255303]',
  'time': '2026-06-24T18:13:08.830Z',
  'level': 'info',
  'message': 'Loading custom processes from /opt/openeo-geopyspark-k8s-custom-processes/src/openeo_geopyspark_k8s_custom_processes/custom_processes.py'},
 {'id': '[1782324788832, 281498]',
  'time': '2026-06-24T18:13:08.832Z',
  'level': 'info',
  'message': 'Overriding process sar_backscatter (namespace backend)'},
 {'id': '[1782324788832, 307622]',
  'time': '2026-06-24T18:13:08.832Z',
  'level': 'info',
  'message': "load_custom_processes: exec loaded '/opt/openeo-geopyspark-k8s-custom-processes/src/openeo_geopyspark_k8s_custom_processes/custom_processes.py'"},
 {'id': '[1782324788832, 816723]',
  'time': '2026-06-24T18:13:08.832Z',
  'level': 'info',
  'message': 'Overriding process sar_backscatter (namespace backend)'},
 {'id': '[1782324803956, 685831]',
  'time': '2026-06-24T18:13:23.956Z',
  'level': 'info',
  'message': 'Job spec: {\n "process_graph": {\n  "loadcollection1": {\n   "process_id": "load_collection",\n   "arguments": {\n    "bands": [\n     "B02",\n     "B03",\n     "B04",\n     "B05",\n     "B07",\n     "B08",\n     "B8A",\n     "B11",\n     "B12"\n    ],\n    "id": "SENTINEL2_L2A",\n    "properties": {\n     "eo:cloud_cover": {\n      "process_graph": {\n       "lte1": {\n        "process_id": "lte",\n        "arguments": {\n         "x": {\n          "from_parameter": "value"\n         },\n         "y": 30\n        },\n        "result": true\n       }\n      }\n     }\n    },\n    "spatial_extent": {\n     "west": -8.23,\n     "south": 40.76,\n     "east": -7.78,\n     "north": 41\n    },\n    "temporal_extent": [\n     "2024-09-30",\n     "2024-10-10"\n    ]\n   }\n  },\n  "aggregatetemporalperiod1": {\n   "process_id": "aggregate_temporal_period",\n   "arguments": {\n    "data": {\n     "from_node": "loadcollection1"\n    },\n    "period": "month",\n    "reducer": {\n     "process_graph": {\n      "median1": {\n       "process_id": "median",\n       "arguments": {\n        "data": {\n         "from_parameter": "data"\n        }\n       },\n       "result": true\n      }\n     }\n    }\n   }\n  },\n  "resamplespatial1": {\n   "process_id": "resample_spatial",\n   "arguments": {\n    "align": "upper-left",\n    "data": {\n     "from_node": "aggregatetemporalperiod1"\n    },\n    "method": "near",\n    "projection": null,\n    "resolution": [\n     20,\n     20\n    ]\n   }\n  },\n  "reducedimension1": {\n   "process_id": "reduce_dimension",\n   "arguments": {\n    "data": {\n     "from_node": "resamplespatial1"\n    },\n    "dimension": "t",\n    "reducer": {\n     "process_graph": {\n      "last1": {\n       "process_id": "last",\n       "arguments": {\n        "data": {\n         "from_parameter": "data"\n        }\n       },\n       "result": true\n      }\n     }\n    }\n   }\n  },\n  "aggregatespatial1": {\n   "process_id": "aggregate_spatial",\n   "arguments": {\n    "data": {\n     "from_node": "reducedimension1"\n    },\n    "geometries": {\n     "type": "FeatureCollection",\n     "features": [\n      {\n       "id": "215",\n       "type": "Feature",\n       "properties": {\n        "label": "Maritime Pines"\n       },\n       "geometry": {\n        "type": "Point",\n        "coordinates": [\n         -7.964046085398656,\n         40.88164564962622\n        ]\n       }\n      },\n      {\n       "id": "189",\n       "type": "Feature",\n       "properties": {\n        "label": "Shrubland"\n       },\n       "geometry": {\n        "type": "Point",\n        "coordinates": [\n         -7.999503387802833,\n         40.94922610

0:00:00 Job 'j-2606251018364139910376c8be04390f': send 'start'
0:00:03 Job 'j-2606251018364139910376c8be04390f': queued (progress 0%)
0:00:08 Job 'j-2606251018364139910376c8be04390f': queued (progress 0%)
0:00:14 Job 'j-2606251018364139910376c8be04390f': queued (progress 0%)
0:00:22 Job 'j-2606251018364139910376c8be04390f': queued (progress 0%)
0:00:32 Job 'j-2606251018364139910376c8be04390f': queued (progress 0%)
0:00:45 Job 'j-2606251018364139910376c8be04390f': running (progress N/A)
0:01:00 Job 'j-2606251018364139910376c8be04390f': running (progress N/A)
0:01:19 Job 'j-2606251018364139910376c8be04390f': running (progress N/A)
0:01:43 Job 'j-2606251018364139910376c8be04390f': running (progress N/A)
0:02:13 Job 'j-2606251018364139910376c8be04390f': finished (progress 100%)


## Visualize labels

In [ ]:
def plot_labels(labels, save_path=None):
    """Plot NFI label points colored by class, with a basemap if contextily is available."""
    labels_wgs84 = labels.to_crs("EPSG:3857")  # web-mercator, needed for basemap tiles

    fig, ax = plt.subplots(figsize=(10, 10))

    # one color per class, plotted as separate layers (so legend works cleanly)
    classes = sorted(labels_wgs84["label"].unique())
    cmap = plt.get_cmap("tab10")
    for i, cls in enumerate(classes):
        subset = labels_wgs84[labels_wgs84["label"] == cls]
        subset.plot(ax=ax, color=cmap(i % 10), markersize=15, label=cls)

    ax.set_title(f"NFI label points (n={len(labels)})")
    ax.legend(title="Label", loc="upper right", fontsize=8)
    ax.set_axis_off()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    # quick class balance printout alongside the plot
    print(labels["label"].value_counts())

plot_labels(labels)